# Price-setter classifier — cell-by-cell walkthrough

This notebook unpacks the data pipeline behind
`classify_price_setter.py` so you can see, for one chosen
`(bus, snapshot)` pair in a solved PyPSA-Eur network, **which** components
are candidate price-setters, what each one's **effective cost at AC** is,
which are **interior** (dispatching strictly inside their bounds), which
**match** the bus price `λ_AC`, and which one ultimately wins.

It deliberately re-uses the helpers from
`classify_price_setter.py` and `plot_price_classification_explainer.py`
rather than re-implementing them, so the walkthrough always tracks the
production logic.

The flow:

1. discover candidates at the bus
2. compute supply / demand / storage-unit cost matrices
3. apply the **interior** mask (dispatch strictly inside bounds)
4. apply the **price-match** mask (`|c_eff − λ| ≤ tol`)
5. intersect → winner
6. cross-check vs `classify(...)` end-to-end
7. visualise the price ladder for this single snapshot


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from matplotlib.gridspec import GridSpec

SCRIPT_DIR = Path.cwd() if Path.cwd().name == "price_formation" else (
    Path.cwd() / "notebooks" / "scripts" / "price_formation"
)
assert SCRIPT_DIR.exists(), f"Cannot find price_formation/ from {Path.cwd()}"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from classify_price_setter import (
    BOUND_TOL_ABS,
    BOUND_TOL_REL,
    CHARGER_CARRIERS,
    DISCHARGER_CARRIERS,
    FLOW_CONGESTION_REL,
    PRICE_TOL_ABS,
    PRICE_TOL_REL,
    build_candidates,
    classify,
    demand_matrices,
    interior_mask,
    load_tech_colors,
    network_path,
    price_match,
    storage_unit_matrices,
    supply_matrices,
    transmission_analysis,
)
from plot_price_classification_explainer import (
    MAX_BANDS,
    candidate_bands_at,
    draw_axis,
    thin_bands,
)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print(f"price_formation dir: {SCRIPT_DIR}")


## Configuration

Change anything in this cell and re-run from here on. Setting `SNAPSHOT = None`
auto-picks a representative snapshot for the chosen `PRESET` from the cached
classifier output (`price_setter_all_buses_<RUN_TAG>.parquet`).

`PRESET` ∈ `{"supply_renewable", "supply_chp", "storage_discharger",
"storage_charger", None}` — `None` means "any classification".


In [ ]:
SCENARIO = "free"
WIGGLE = 1000
HIKE = None

BUS = "DE0 0"
SNAPSHOT = None
PRESET = "supply_chp"

RUN_TAG = f"{SCENARIO}_{WIGGLE}" + (f"_{HIKE}" if HIKE is not None else "")
PARQUET = SCRIPT_DIR / f"price_setter_all_buses_{RUN_TAG}.parquet"
print(f"RUN_TAG = {RUN_TAG}")
print(f"BUS     = {BUS}")
print(f"PRESET  = {PRESET}")
print(f"cache   = {PARQUET}  (exists={PARQUET.exists()})")


## Load the network and resolve `(BUS, SNAPSHOT)`

If `SNAPSHOT is None`, we pull the cached classifier output and pick the
row with the largest `|λ|` in the `(BUS, PRESET)` slice — same selection
rule the explainer figure uses, just constrained to one bus.


In [ ]:
path = network_path(scenario=SCENARIO, wiggle=WIGGLE, hike=HIKE)
print(f"Loading {path.name}")
n = pypsa.Network(path)
print(f"  {len(n.buses)} buses, {len(n.snapshots)} snapshots, "
      f"AC buses: {(n.buses.carrier == 'AC').sum()}")

PRESET_TARGETS = {
    "supply_renewable": ("supply",
                         {"onwind", "solar-hsat", "offwind-dc", "solar",
                          "offwind-ac", "ror"}),
    "supply_chp":       ("supply",
                         {"urban central solid biomass CHP",
                          "urban central gas CHP", "waste CHP"}),
    "storage_discharger": ("storage_discharger",
                           {"battery discharger", "home battery discharger",
                            "V2G"}),
    "storage_charger":  ("storage_charger",
                         {"battery charger", "home battery charger",
                          "BEV charger"}),
}

def pick_snapshot(table: pd.DataFrame, bus: str, preset: str | None):
    sub = table[table["bus"] == bus]
    if preset is not None and preset in PRESET_TARGETS:
        cls, carriers = PRESET_TARGETS[preset]
        target = sub[(sub["classification"] == cls)
                     & (sub["price_setter_carrier"].isin(carriers))]
        if target.empty:
            target = sub[sub["classification"] == cls]
        if target.empty:
            print(f"  no rows for preset={preset!r} at bus={bus!r}; "
                  f"falling back to any row at bus")
            target = sub
    else:
        target = sub
    if target.empty:
        raise SystemExit(f"No rows at bus={bus!r}")
    idx = target["price_eur_per_mwh"].abs().idxmax()
    return pd.Timestamp(target.loc[idx, "snapshot"])

if SNAPSHOT is None:
    if not PARQUET.exists():
        raise SystemExit(
            f"No cache at {PARQUET}; run classify_price_setter.py first or "
            f"set SNAPSHOT explicitly."
        )
    table = pd.read_parquet(PARQUET)
    SNAPSHOT = pick_snapshot(table, BUS, PRESET)
    print(f"  auto-picked snapshot: {SNAPSHOT}")
else:
    SNAPSHOT = pd.Timestamp(SNAPSHOT)
    print(f"  using explicit snapshot: {SNAPSHOT}")

LAM = float(n.buses_t.marginal_price.at[SNAPSHOT, BUS])
loads = n.loads.index[n.loads.bus == BUS]
load_at_t = float(
    n.loads_t.p_set.reindex(columns=loads, index=[SNAPSHOT])
    .fillna(0.0).iloc[0].sum()
) if len(loads) else 0.0

print()
print(f"BUS         = {BUS}")
print(f"SNAPSHOT    = {SNAPSHOT}")
print(f"λ_AC(b, t)  = {LAM:.3f} EUR/MWh")
print(f"load(b, t)  = {load_at_t:.1f} MW")


## Step 1 — Discover candidates

`build_candidates(n, bus)` returns four DataFrames listing **every** local
component whose price could conceivably set `λ_AC` at `bus`:

- **generators** at the bus (e.g. wind, solar, ror)
- **supply links** with `bus1 == bus` (e.g. CCGT, CHP, dischargers).
  `role ∈ {supply, discharger}`.
- **demand links** with `bus0 ∈ {bus, "{bus} low voltage"}`
  (e.g. electrolyser, heat pump, battery charger).
  `role ∈ {demand_flex, charger}`, plus an `on_lv` flag.
- **storage units** at the bus (hydro, PHS).

Pure-topology wires (`DC`, `electricity distribution grid`) are excluded;
they are not capable of *setting* a price.


In [ ]:
gens, supply_links, demand_links, sus = build_candidates(n, BUS)

print(f"generators     : {len(gens):3d}")
print(f"supply links   : {len(supply_links):3d}  "
      f"(supply={(supply_links.role == 'supply').sum()}, "
      f"discharger={(supply_links.role == 'discharger').sum()})")
print(f"demand links   : {len(demand_links):3d}  "
      f"(demand_flex={(demand_links.role == 'demand_flex').sum()}, "
      f"charger={(demand_links.role == 'charger').sum()}, "
      f"on_lv={demand_links.on_lv.sum()})")
print(f"storage units  : {len(sus):3d}")


In [ ]:
# Per-carrier breakdown
def carrier_counts(df, role_col=None):
    if df.empty:
        return df
    cols = ["carrier"] + ([role_col] if role_col else [])
    return df.groupby(cols).size().rename("count").to_frame()

print("Generators at bus:")
display(carrier_counts(gens))

print("Supply links (bus1 == bus):")
display(carrier_counts(supply_links, "role"))

print("Demand links (bus0 == bus or bus low voltage):")
display(carrier_counts(demand_links, "role"))

print("Storage units at bus:")
display(carrier_counts(sus))


## Step 2 — Supply cost matrix

For each supply candidate we build the **effective cost at AC**, $c_\mathrm{eff}$:

- **Generator at bus**: $c_\mathrm{eff}(t) = \mathrm{marginal\_cost}(t)$
  (essentially zero for renewables).
- **Single-output supply Link** (input fuel at `bus0`, AC at `bus1`,
  efficiency $\eta$):
  $$c_\mathrm{eff}(t) \;=\; \frac{c_\mathrm{link}(t) + \lambda_{\mathrm{bus0}}(t)}{\eta(t)}$$
- **CHP** (also produces heat at `bus2`, optionally CO₂ at `bus3`):
  $$c_\mathrm{eff}(t) \;=\; \frac{c_\mathrm{link} + \lambda_{\mathrm{bus0}} - \eta_2\lambda_{\mathrm{bus2}} - \eta_3\lambda_{\mathrm{bus3}}}{\eta_1}$$

Without the `bus2`/`bus3` correction, biomass and gas CHPs always look
overpriced and never get classified as marginal even when they obviously are.

We call `supply_matrices(...)` for the **single snapshot** of interest, then
slice each `(T=1, K)` frame at that snapshot to get a per-candidate row.


In [ ]:
snapshots = pd.Index([SNAPSHOT])
c_s, p_s, pn_s, pmax_s, pmin_s, meta_s = supply_matrices(
    n, gens, supply_links, snapshots
)

if c_s.empty:
    print("No supply candidates at this bus.")
    supply_table = pd.DataFrame()
else:
    supply_table = pd.DataFrame({
        "carrier":   meta_s["carrier"],
        "ctype":     meta_s["ctype"],
        "role":      meta_s["role"],
        "fuel_bus":  meta_s["fuel_bus"],
        "c_eff":     c_s.loc[SNAPSHOT].astype(float),
        "p_AC":      p_s.loc[SNAPSHOT].astype(float),
        "p_min":     (pn_s.loc[SNAPSHOT] * pmin_s.loc[SNAPSHOT]).astype(float),
        "p_max":     (pn_s.loc[SNAPSHOT] * pmax_s.loc[SNAPSHOT]).astype(float),
    })
    supply_table["abs_diff_lam"] = (supply_table["c_eff"] - LAM).abs()
    supply_table = supply_table.sort_values("abs_diff_lam")

print(f"λ_AC = {LAM:.3f}")
print(f"Supply candidates ranked by |c_eff − λ| (closest at top):")
display(supply_table.head(20))


## Step 3 — Demand cost matrix

For each demand-flex candidate (an interior demand link is willing to pay
exactly its **value of consumption**, otherwise it would consume more or less):

$$c_\mathrm{eff}(t) \;=\; \eta\,\lambda_{\mathrm{bus1}}(t) \;+\; \eta_2\lambda_{\mathrm{bus2}} \;+\; \eta_3\lambda_{\mathrm{bus3}} \;-\; c_\mathrm{link}(t)$$

For LV-side links (`bus0 == "{bus} low voltage"`) the same formula applies
because the distribution-grid efficiency is ≈ 1, so LV ≈ AC in price.


In [ ]:
c_d, p_d, pn_d, pmax_d, pmin_d, meta_d = demand_matrices(
    n, demand_links, snapshots
)

if c_d.empty:
    print("No demand-flex candidates at this bus.")
    demand_table = pd.DataFrame()
else:
    demand_table = pd.DataFrame({
        "carrier":   meta_d["carrier"],
        "role":      meta_d["role"],
        "sink_bus":  meta_d["sink_bus"],
        "on_lv":     meta_d["on_lv"],
        "c_eff":     c_d.loc[SNAPSHOT].astype(float),
        "p_with":    p_d.loc[SNAPSHOT].astype(float),
        "p_min":     (pn_d.loc[SNAPSHOT] * pmin_d.loc[SNAPSHOT]).astype(float),
        "p_max":     (pn_d.loc[SNAPSHOT] * pmax_d.loc[SNAPSHOT]).astype(float),
    })
    demand_table["abs_diff_lam"] = (demand_table["c_eff"] - LAM).abs()
    demand_table = demand_table.sort_values("abs_diff_lam")

print(f"λ_AC = {LAM:.3f}")
print(f"Demand candidates ranked by |c_eff − λ| (closest at top):")
display(demand_table.head(20))


## Step 4 — Storage units (hydro / PHS)

`StorageUnit` encapsulates both modes (charging and discharging) on **one**
component, with `p` signed (positive = discharging, negative = charging). The
classifier treats an SU as price-setting whenever it is dispatching strictly
inside its bounds — its `c_eff` is taken to be the **water value**, which by
LP optimality just equals `λ_AC` at this bus.


In [ ]:
p_su, pn_su, meta_su = storage_unit_matrices(n, sus, snapshots)

if p_su.empty:
    print("No storage units at this bus.")
    su_table = pd.DataFrame()
else:
    p_row = p_su.loc[SNAPSHOT].astype(float)
    pn_row = pn_su.loc[SNAPSHOT].astype(float)
    bound_tol = np.maximum(BOUND_TOL_ABS, BOUND_TOL_REL * pn_row.abs())
    interior_su = (p_row.abs() > bound_tol) & (p_row.abs() < pn_row.abs() - bound_tol)
    su_table = pd.DataFrame({
        "carrier":  meta_su["carrier"],
        "p":        p_row,
        "p_nom":    pn_row,
        "interior": interior_su,
    })

display(su_table)


## Step 5 — The interior mask

A candidate can only set the price if it is **strictly between its bounds**
at this snapshot — at a corner, the LP could push more or less without
changing the objective at this rate.

```
interior = (p > p_nom·p_min_pu + tol)  AND  (p < p_nom·p_max_pu − tol)
tol      = max(BOUND_TOL_ABS, BOUND_TOL_REL · p_nom)
       = max({BOUND_TOL_ABS}, {BOUND_TOL_REL} · p_nom)
```

We apply `interior_mask(...)` to both the supply and demand frames, then
visualise each candidate's dispatch as a *fraction* of its capacity range.
A candidate is interior whenever its bar is strictly between the two grey
margins.


In [ ]:
def interior_at(p, pn, pmin, pmax):
    if p.empty:
        return pd.Series(dtype=bool)
    return interior_mask(p, pn, pmin, pmax).loc[SNAPSHOT]

int_supply = interior_at(p_s, pn_s, pmin_s, pmax_s)
int_demand = interior_at(p_d, pn_d, pmin_d, pmax_d)

print(f"interior supply : {int(int_supply.sum())} / {len(int_supply)}")
print(f"interior demand : {int(int_demand.sum())} / {len(int_demand)}")
print()
print("Interior supply candidates:")
display(supply_table.loc[supply_table.index.intersection(
    int_supply.index[int_supply])].head(20))
print("Interior demand candidates:")
display(demand_table.loc[demand_table.index.intersection(
    int_demand.index[int_demand])].head(20))


In [ ]:
# Visualise dispatch fraction for every candidate (supply + demand)
def fraction_at(p, pn, pmin, pmax):
    if p.empty:
        return pd.DataFrame()
    lo = (pn * pmin).loc[SNAPSHOT]
    hi = (pn * pmax).loc[SNAPSHOT]
    pp = p.loc[SNAPSHOT]
    span = (hi - lo).replace(0, np.nan)
    frac = ((pp - lo) / span).clip(-0.05, 1.05)
    tol = np.maximum(BOUND_TOL_ABS, BOUND_TOL_REL * pn.loc[SNAPSHOT].abs())
    tol_frac = (tol / span).fillna(0.0)
    return pd.DataFrame({"frac": frac, "tol_frac": tol_frac, "p_nom": pn.loc[SNAPSHOT]})

fs = fraction_at(p_s, pn_s, pmin_s, pmax_s)
fd = fraction_at(p_d, pn_d, pmin_d, pmax_d)

# keep only candidates whose carrier dispatches non-negligibly so the
# plot is readable even on big buses
def keep_active(frame, p, pn):
    if frame.empty:
        return frame
    pp = p.loc[SNAPSHOT].abs()
    pn_v = pn.loc[SNAPSHOT].abs()
    active = (pp > np.maximum(BOUND_TOL_ABS, BOUND_TOL_REL * pn_v))
    return frame[active]

fs = keep_active(fs, p_s, pn_s)
fd = keep_active(fd, p_d, pn_d)

n_rows = max(len(fs), 1) + max(len(fd), 1)
fig, axes = plt.subplots(2, 1, figsize=(11, 0.35 * n_rows + 1.2),
                         gridspec_kw={"height_ratios":
                                      [max(len(fs), 1), max(len(fd), 1)]})
for ax, frame, title, mask in [
    (axes[0], fs, "supply (interior = bar strictly between grey margins)", int_supply),
    (axes[1], fd, "demand-flex / chargers", int_demand),
]:
    if frame.empty:
        ax.text(0.5, 0.5, f"no active {title.split()[0]} candidates",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        continue
    y = np.arange(len(frame))
    is_int = mask.reindex(frame.index, fill_value=False).values
    ax.barh(y, frame["frac"].values, color=["#1f77b4" if i else "#aaaaaa"
                                            for i in is_int],
            edgecolor="black", lw=0.4)
    # show interior tolerance as shaded zones at 0 and 1
    for yi, tf in zip(y, frame["tol_frac"].values):
        ax.axhspan(yi - 0.4, yi + 0.4, xmin=0, xmax=tf, color="#dddddd",
                   alpha=0.7, zorder=0)
        ax.axhspan(yi - 0.4, yi + 0.4, xmin=1 - tf, xmax=1, color="#dddddd",
                   alpha=0.7, zorder=0)
    ax.axvline(0, color="black", lw=0.5)
    ax.axvline(1, color="black", lw=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{nm}" for nm in frame.index], fontsize=8)
    ax.set_xlim(-0.05, 1.05)
    ax.set_xlabel("dispatch as fraction of (p_min, p_max)")
    ax.set_title(title, fontsize=9, loc="left")
    ax.invert_yaxis()
fig.tight_layout()
plt.show()


## Step 6 — The price-match mask

An interior candidate's `c_eff` should equal `λ_AC` exactly under LP
optimality. In practice the solver leaves a small numerical residual, so we
require:

```
match = |c_eff − λ| ≤ max(PRICE_TOL_ABS, PRICE_TOL_REL · |λ|)
```

with `PRICE_TOL_ABS = 0.5 EUR/MWh`, `PRICE_TOL_REL = 0.01`.


In [ ]:
tol_lam = max(PRICE_TOL_ABS, PRICE_TOL_REL * abs(LAM))
print(f"|λ| = {abs(LAM):.3f}, match tolerance = {tol_lam:.3f} EUR/MWh")
print()

def match_at(c_eff):
    if c_eff.empty:
        return pd.Series(dtype=bool)
    return price_match(c_eff, n.buses_t.marginal_price[BUS]
                       .reindex(c_eff.index)).loc[SNAPSHOT]

m_supply = match_at(c_s)
m_demand = match_at(c_d)

print(f"matching supply : {int(m_supply.sum())} / {len(m_supply)}")
print(f"matching demand : {int(m_demand.sum())} / {len(m_demand)}")
print()
print("Top 10 supply candidates by |c_eff − λ|:")
display(supply_table[["carrier", "role", "c_eff", "abs_diff_lam"]]
        .head(10).assign(within_tol=lambda d: d["abs_diff_lam"] <= tol_lam))
print("Top 10 demand candidates by |c_eff − λ|:")
display(demand_table[["carrier", "role", "c_eff", "abs_diff_lam"]]
        .head(10).assign(within_tol=lambda d: d["abs_diff_lam"] <= tol_lam))


## Step 7 — Combined: interior **and** match → winner

The price setter is the candidate that is simultaneously interior and
matching, with the smallest `|c_eff − λ|`. We compute that here, then
compare to the cached classifier result for the same row.


In [ ]:
def best_winner(table, interior_mask_, match_mask_):
    if table.empty:
        return None
    cand = table.copy()
    cand["interior"] = interior_mask_.reindex(cand.index, fill_value=False)
    cand["match"] = match_mask_.reindex(cand.index, fill_value=False)
    eligible = cand[cand["interior"] & cand["match"]]
    if eligible.empty:
        return None
    return eligible.sort_values("abs_diff_lam").iloc[0]

w_s = best_winner(supply_table, int_supply, m_supply)
w_d = best_winner(demand_table, int_demand, m_demand)

print("Supply cascade winner:")
print(w_s if w_s is not None else "  (none — no supply candidate is interior+matching)")
print()
print("Demand cascade winner:")
print(w_d if w_d is not None else "  (none)")
print()

# Priority: supply > demand_flex > storage_unit > transmission > load_shed
if w_s is not None:
    print(f"\nManual classification: '{w_s['role']}' (priority: supply cascade)")
    print(f"  carrier  = {w_s['carrier']}")
    print(f"  c_eff    = {w_s['c_eff']:.3f}   (λ = {LAM:.3f},  diff = {w_s['abs_diff_lam']:.3f})")
elif w_d is not None:
    print(f"\nManual classification: '{w_d['role']}' (priority: demand cascade)")
    print(f"  carrier  = {w_d['carrier']}")
    print(f"  c_eff    = {w_d['c_eff']:.3f}   (λ = {LAM:.3f},  diff = {w_d['abs_diff_lam']:.3f})")
else:
    print("\nNo strict winner — would fall through to storage_unit / transmission / load_shed / unresolved.")


## Step 8 — Transmission and load-shedding fallbacks

If no local Link is interior+matching, the classifier asks: is an incident
line saturated *into* this bus, with a cheaper price on the other side? If
so, the price is set by **import**. Below we run `transmission_analysis`
and inspect the most-loaded incident line at this snapshot.


In [ ]:
cong = transmission_analysis(n, BUS, snapshots)
display(cong)
print()
if not cong.empty and not pd.isna(cong.at[SNAPSHOT, "flow_frac"]):
    frac = cong.at[SNAPSHOT, "flow_frac"]
    op = cong.at[SNAPSHOT, "other_price"]
    print(f"Most-loaded inbound line at {SNAPSHOT}: {cong.at[SNAPSHOT, 'name']}")
    print(f"  flow fraction          = {frac*100:.1f}% of capacity "
          f"(threshold {FLOW_CONGESTION_REL*100:.0f}%)")
    print(f"  other-side price       = {op:.3f} EUR/MWh  (here: {LAM:.3f})")
    saturated = frac >= FLOW_CONGESTION_REL
    cheap_neighbour = (not pd.isna(op)) and (op <= LAM + PRICE_TOL_ABS)
    if saturated and cheap_neighbour:
        print(f"  → would classify as 'transmission' if no local cascade matched.")
    else:
        print(f"  → not a transmission case (saturated={saturated}, cheap_neighbour={cheap_neighbour}).")


In [ ]:
# Load-shedding: any slack generator (marginal_cost >= VoLL_THRESH) actually
# dispatching here?
from classify_price_setter import VOLL_THRESH
shed_gens = n.generators[(n.generators.bus == BUS)
                         & (n.generators.marginal_cost >= VOLL_THRESH)]
if len(shed_gens):
    p_shed = (n.generators_t.p
              .reindex(columns=shed_gens.index, index=[SNAPSHOT])
              .fillna(0.0).iloc[0])
    print(f"Slack/VoLL generators at bus (marginal_cost >= {VOLL_THRESH}):")
    display(pd.DataFrame({
        "marginal_cost": shed_gens["marginal_cost"],
        "p_at_t": p_shed,
        "active": p_shed > 1e-3,
    }))
else:
    print(f"No slack/VoLL generator at {BUS}.")


## Step 9 — End-to-end: `classify(n, BUS)`

Same machinery, but executed across **all** snapshots in one call. We pull
the row at `SNAPSHOT` and compare against the manual derivation above —
they should agree.


In [ ]:
classified = classify(n, BUS)
row = classified.loc[SNAPSHOT]
print("classify(n, BUS) row at this snapshot:")
display(row.to_frame("value"))

print()
print("Sanity check — manual vs classify():")
manual_winner = (w_s if w_s is not None else w_d)
if manual_winner is not None:
    manual_carrier = manual_winner["carrier"]
    manual_class = manual_winner["role"]
    cls_match = "supply" if manual_class == "supply" else (
        "storage_discharger" if manual_class == "discharger" else (
            "storage_charger" if manual_class == "charger" else "demand_flex"))
    ok_class = (row["classification"] == cls_match)
    ok_carrier = (row["price_setter_carrier"] == manual_carrier)
    print(f"  manual class    = {cls_match!r:<22}  classify = {row['classification']!r}  "
          f"{'OK' if ok_class else 'MISMATCH'}")
    print(f"  manual carrier  = {manual_carrier!r:<22}  classify = "
          f"{row['price_setter_carrier']!r}  {'OK' if ok_carrier else 'MISMATCH'}")
else:
    print(f"  manual winner = (none).  classify = {row['classification']!r} "
          f"{row['price_setter_carrier']!r}")

print()
print("Explanation written by classify():")
print(f"  {row['explanation']}")


## Step 10 — Visualise the price ladder for this snapshot

This reuses the explainer's drawing helpers (`candidate_bands_at`,
`thin_bands`, `draw_axis`) to render a single-cell version of the figure
in `plot_price_classification_explainer.py`. The vertical bar on the left
shows every interior candidate's `c_eff` as a coloured band; the right
column derives `c_eff` from the actual numbers in the network. The
horizontal black line marks the **observed** `λ_AC`.


In [ ]:
tech_colors = load_tech_colors()
bands = candidate_bands_at(n, BUS, SNAPSHOT)
print(f"interior candidates at this snapshot: {len(bands)}")

winning_kind = row["classification"]
winning_carrier = row["price_setter_carrier"]
bands = thin_bands(bands, MAX_BANDS, LAM, winning_kind, winning_carrier)

fig = plt.figure(figsize=(13, 7))
outer = GridSpec(1, 1, figure=fig)
title = (f"{BUS} · {SNAPSHOT:%Y-%m-%d %H:%M} · "
         f"observed: {winning_carrier} ({winning_kind})")
draw_axis(fig, outer[0, 0], title, bands, LAM,
          winning_kind, winning_carrier, tech_colors)
plt.show()


## Step 11 — Time-series context

Where does this snapshot sit relative to the rest of the year? The plot below
shows `λ_AC(BUS, t)` for a 7-day window centred on `SNAPSHOT`, with points
coloured by classification (using the cached parquet) and a vertical marker
at our chosen snapshot.


In [ ]:
window = pd.Timedelta(days=3, hours=12)
table = pd.read_parquet(PARQUET) if PARQUET.exists() else pd.DataFrame()
sub = table[table["bus"] == BUS].copy() if not table.empty else pd.DataFrame()
if not sub.empty:
    sub["snapshot"] = pd.to_datetime(sub["snapshot"])
    sub = sub.set_index("snapshot").sort_index()
    sub = sub.loc[SNAPSHOT - window: SNAPSHOT + window]

CLASS_FALLBACK_COLOR = {
    "transmission": "#555555",
    "unresolved": "#bbbbbb",
    "load_shedding": "#ff0040",
    "storage_unit": "#2b8cbe",
}

def color_for_row(r):
    car = r["price_setter_carrier"]
    if r["classification"] in CLASS_FALLBACK_COLOR and (
        not car or car not in tech_colors
    ):
        return CLASS_FALLBACK_COLOR[r["classification"]]
    return tech_colors.get(car, CLASS_FALLBACK_COLOR["unresolved"])

def label_for_row(r):
    return (f"{r['classification']}: {r['price_setter_carrier']}"
            if r["price_setter_carrier"] else r["classification"])

fig, ax = plt.subplots(figsize=(13, 4))
if not sub.empty:
    colors = [color_for_row(r) for _, r in sub.iterrows()]
    ax.plot(sub.index, sub["price_eur_per_mwh"],
            color="lightgrey", lw=0.8, zorder=1)
    ax.scatter(sub.index, sub["price_eur_per_mwh"],
               c=colors, s=30, edgecolor="black", linewidth=0.3, zorder=2)

    # Legend: one entry per (classification, carrier) actually present
    from matplotlib.patches import Patch
    seen = {}
    for (_, r), col in zip(sub.iterrows(), colors):
        seen.setdefault(label_for_row(r), col)
    handles = [Patch(facecolor=c, edgecolor="black", label=l)
               for l, c in seen.items()]
    ax.legend(handles=handles, loc="center left",
              bbox_to_anchor=(1.005, 0.5), frameon=True, fontsize=8,
              title="price setter")
else:
    lam = n.buses_t.marginal_price[BUS]
    win = lam.loc[SNAPSHOT - window: SNAPSHOT + window]
    ax.plot(win.index, win.values, color="black", lw=0.8)
ax.axvline(SNAPSHOT, color="red", lw=1.0, ls="--", zorder=3)
ax.axhline(LAM, color="red", lw=0.5, alpha=0.5, zorder=0)
ax.set_ylabel(f"λ_AC({BUS}) [EUR/MWh]")
ax.set_title(f"7-day window around {SNAPSHOT:%Y-%m-%d %H:%M} "
             f"(red dashed = chosen snapshot, λ = {LAM:.2f} EUR/MWh)",
             fontsize=10, loc="left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()


## Step 12 — Heat-side correction diagnostic for biomass CHP

A back-pressure CHP has a single dispatch variable (`p0` = fuel), so its
KKT condition at interior dispatch is

\[
\eta_1\,\lambda_{AC} \;+\; \eta_2\,\lambda_{heat} \;=\; \lambda_{fuel} + c
\]

i.e. it is **jointly** marginal for AC and heat. The classifier therefore
computes `c_eff_AC` *with* the heat-side term:

\[
c_{eff,AC} \;=\; \frac{\lambda_{fuel} + c - \eta_2\,\lambda_{heat}}{\eta_1}
\]

If we instead used the naive `(λ_fuel + c) / η₁` we'd often *miss*
the price match — biomass CHPs would look too expensive at all hours.
This cell quantifies how much the heat correction is doing in practice for
the biomass CHP at this bus.


In [ ]:
# Find the biomass CHP at this bus, if any
bio_chp_links = n.links[
    (n.links.bus1 == BUS)
    & n.links.carrier.str.contains("biomass CHP", case=False, na=False)
]
if bio_chp_links.empty:
    print(f"No biomass CHP at {BUS}. Pick a bus with central biomass CHP "
          f"(most northern/central European nodes have one).")
else:
    nm = bio_chp_links.index[0]
    b0 = n.links.at[nm, "bus0"]
    b2 = n.links.at[nm, "bus2"]
    eta1 = float(n.links.at[nm, "efficiency"])
    eta2 = float(n.links.at[nm, "efficiency2"])
    c = float(n.links.at[nm, "marginal_cost"])
    print(f"Component:      {nm}")
    print(f"  bus0 (fuel) = {b0}")
    print(f"  bus2 (heat) = {b2}")
    print(f"  η₁ = {eta1:.4f},  η₂ = {eta2:.4f},  c = {c:.3f} EUR/MWh")

    # All hours where this carrier was tagged as the price setter at this bus
    chp_rows = table[
        (table["bus"] == BUS)
        & (table["price_setter_carrier"] == n.links.at[nm, "carrier"])
    ].copy()
    chp_rows["snapshot"] = pd.to_datetime(chp_rows["snapshot"])
    chp_rows = chp_rows.set_index("snapshot").sort_index()
    print(f"  hours tagged as setter: {len(chp_rows)} / {len(table[table['bus']==BUS])}")

    if not chp_rows.empty:
        chp_rows["lam_fuel"] = n.buses_t.marginal_price.loc[
            chp_rows.index, b0
        ].values
        chp_rows["lam_heat"] = n.buses_t.marginal_price.loc[
            chp_rows.index, b2
        ].values
        chp_rows["c_eff_no_heat"] = (c + chp_rows["lam_fuel"]) / eta1
        chp_rows["c_eff_with_heat"] = (
            c + chp_rows["lam_fuel"] - eta2 * chp_rows["lam_heat"]
        ) / eta1
        chp_rows["err_no_heat"] = (
            chp_rows["c_eff_no_heat"] - chp_rows["price_eur_per_mwh"]
        ).abs()
        chp_rows["err_with_heat"] = (
            chp_rows["c_eff_with_heat"] - chp_rows["price_eur_per_mwh"]
        ).abs()

        print()
        print("Distribution across CHP-as-setter hours:")
        display(chp_rows[[
            "price_eur_per_mwh", "lam_fuel", "lam_heat",
            "c_eff_no_heat", "c_eff_with_heat",
            "err_no_heat", "err_with_heat",
        ]].describe().round(3))

        print()
        tol = max(PRICE_TOL_ABS, PRICE_TOL_REL * chp_rows["price_eur_per_mwh"].abs().median())
        match_no = (chp_rows["err_no_heat"] <= tol).mean()
        match_yes = (chp_rows["err_with_heat"] <= tol).mean()
        print(f"Share of CHP-setter hours where |c_eff − λ| ≤ ~{tol:.2f} EUR/MWh:")
        print(f"  with    heat correction (used by classifier): {match_yes*100:5.1f}%")
        print(f"  without heat correction (naive fuel/η₁)        : {match_no*100:5.1f}%")
        print()
        print("Interpretation: if the 'with heat' rate is much higher than the")
        print("'without heat' rate, the bus2 correction is doing real work — the")
        print("CHP looks too expensive on its electricity alone but is actually")
        print("marginal because the LP credits the joint heat output.")

        # Visualise the two c_eff series against λ_AC
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(chp_rows.index, chp_rows["price_eur_per_mwh"],
                color="black", lw=0.8, label=r"$\lambda_{AC}$")
        ax.scatter(chp_rows.index, chp_rows["c_eff_with_heat"],
                   s=14, color="#1f77b4", alpha=0.7,
                   label=r"$c_{eff}$ with heat correction")
        ax.scatter(chp_rows.index, chp_rows["c_eff_no_heat"],
                   s=14, color="#d62728", alpha=0.4,
                   label=r"$c_{eff}$ without heat correction")
        ax.set_ylabel("EUR/MWh")
        ax.set_title(f"{nm}: c_eff with vs. without heat-side correction "
                     f"on hours where it sets the price",
                     fontsize=10, loc="left")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.yaxis.grid(True, alpha=0.3)
        ax.set_axisbelow(True)
        ax.legend(loc="best", fontsize=8)
        fig.tight_layout()
        plt.show()


## Step 13 — Try another preset

The whole walkthrough is parameterised on the configuration cell. To see
a different case, change `PRESET` (or `BUS` / `SNAPSHOT`) up there and
re-run from the configuration cell onward:

- `PRESET = "supply_renewable"` — wind / solar / RoR setting price.
- `PRESET = "supply_chp"` — gas / biomass CHP, exercises the bus2/bus3
  correction.
- `PRESET = "storage_discharger"` — battery / V2G discharger sets price
  via its water value.
- `PRESET = "storage_charger"` — battery / EV charger sets price as
  flexible demand.
- `PRESET = None` — pick the snapshot with the highest `|λ|` regardless
  of class.

Hand-setting an `SNAPSHOT` whose classifier row is `unresolved` is also
informative — Step 7 will report no winner, and Step 9 will agree.
